In [21]:
%pip install wordfreq

Note: you may need to restart the kernel to use updated packages.


In [22]:
from wordfreq import top_n_list, zipf_frequency
import re
import random



def clean_token(w: str) -> str:
    return re.sub(r"[^a-z]", "", w.lower())



def load_vocab(lang: str = "en", n: int = 100_000, min_len: int = 3) -> set[str]:
    raw_words = top_n_list(lang, n)
    cleaned = {clean_token(w) for w in raw_words}
    return {w for w in cleaned if len(w) >= min_len}

vocab = load_vocab("en", n=100_000)
print(f"Loaded {len(vocab)} words into vocabulary.")

Loaded 95292 words into vocabulary.


In [23]:
from collections import defaultdict
from typing import Optional

MIN_ZIPF_CLUE = 3.0
MIN_ZIPF_COMPOUND = 2.5

BANNED_PREFIX_FRAGMENTS = {
    "pre", "re", "un", "sub", "trans", "inter", "intra", "micro", "macro", "ultra",
    "semi", "co", "com", "con", "non", "anti", "post", "pro", "sub", "mis", "dis",
    "over", "under", "auto", "bio", "geo", "tele", "tri", "bi", "mono", "uni",
}
BANNED_SUFFIX_FRAGMENTS = {
    "tion", "sion", "ness", "ment", "ism", "ist", "ity", "ship", "hood", "ous",
    "able", "ible", "less", "ful", "ish", "ive", "ize", "ise", "er", "est",
}

def is_banned_fragment(fragment: str, banned: set[str]) -> bool:
    return fragment in banned

def find_prefix_group(base_word: str,
                      vocab: set[str],
                      min_prefix_len: int = 2,
                      max_prefix_len: int = 8,
                      min_group_size: int = 4) -> list[str]:
    base = clean_token(base_word)
    results: set[str] = set()

    for w in vocab:
        if not w.endswith(base):
            continue
        if len(w) <= len(base) + min_prefix_len - 1:
            continue
        prefix = w[:-len(base)]
        if not (min_prefix_len <= len(prefix) <= max_prefix_len):
            continue
        if prefix not in vocab:
            continue
        if is_banned_fragment(prefix, BANNED_PREFIX_FRAGMENTS):
            continue
        if zipf_frequency(prefix, "en") < MIN_ZIPF_CLUE:
            continue
        if zipf_frequency(w, "en") < MIN_ZIPF_COMPOUND:
            continue
        results.add(prefix)

    ordered = sorted(results)
    if len(ordered) < min_group_size:
        return []
    return ordered

def find_suffix_group(base_word: str,
                      vocab: set[str],
                      min_suffix_len: int = 2,
                      max_suffix_len: int = 8,
                      min_group_size: int = 4) -> list[str]:
    base = clean_token(base_word)
    results: set[str] = set()

    for w in vocab:
        if not w.startswith(base):
            continue
        if len(w) <= len(base) + min_suffix_len - 1:
            continue

        suffix = w[len(base):]
        if not (min_suffix_len <= len(suffix) <= max_suffix_len):
            continue
        if suffix not in vocab:
            continue
        if is_banned_fragment(suffix, BANNED_SUFFIX_FRAGMENTS):
            continue
        if zipf_frequency(suffix, "en") < MIN_ZIPF_CLUE:
            continue
        if zipf_frequency(w, "en") < MIN_ZIPF_COMPOUND:
            continue

        results.add(suffix)

    ordered = sorted(results)
    if len(ordered) < min_group_size:
        return []
    return ordered

def generate_prefix_puzzle(base_word: str,
                           vocab: set[str],
                           n_clues: int = 4,
                           rng: Optional[random.Random] = None) -> tuple[list[str], str]:
    rng = rng or random
    group = find_prefix_group(base_word, vocab, min_group_size=n_clues)
    if not group:
        raise ValueError(f"No sufficient prefix group found for base word '{base_word}'")

    clues = rng.sample(group, n_clues)
    return clues, base_word

def generate_suffix_puzzle(base_word: str,
                           vocab: set[str],
                           n_clues: int = 4,
                           rng: Optional[random.Random] = None) -> tuple[list[str], str]:
    rng = rng or random
    group = find_suffix_group(base_word, vocab, min_group_size=n_clues)
    if not group:
        raise ValueError(f"No sufficient suffix group found for base word '{base_word}'")

    clues = rng.sample(group, n_clues)
    return clues, base_word

In [24]:
rng = random.Random(0)
bases_prefix = ["man", "light", "star", "dog"]
bases_suffix = ["dog", "house", "time", "book"]



print("Prefix-based puzzles (___ CATEGORY):")
for base in bases_prefix:
    try:
        clues, category = generate_prefix_puzzle(base, vocab, n_clues=4, rng=rng)
        print(f"  Clues: {clues}  |  Category: ___ {category.upper()}")
    except ValueError:
        pass

print("\nSuffix-based puzzles (CATEGORY ___):")

for base in bases_suffix:
    try:
        clues, category = generate_suffix_puzzle(base, vocab, n_clues=4, rng=rng)
        print(f"  Clues: {clues}  |  Category: {category.upper()} ___")
    except ValueError:
        pass

try:
    man_clues, man_category = generate_prefix_puzzle("man", vocab, n_clues=4, rng=rng)
    print("\nExample for 'MAN':")
    print(f"  Clues: {man_clues}  |  Category: ___ {man_category.upper()}")

except ValueError as e:
    print(e)

Prefix-based puzzles (___ CATEGORY):
  Clues: ['marks', 'patrol', 'bat', 'gun']  |  Category: ___ MAN
  Clues: ['moon', 'lime', 'high', 'green']  |  Category: ___ LIGHT
  Clues: ['north', 'morning', 'porn', 'battle']  |  Category: ___ STAR

Suffix-based puzzles (CATEGORY ___):
  Clues: ['house', 'wood', 'fight', 'matic']  |  Category: DOG ___
  Clues: ['mates', 'holds', 'mate', 'work']  |  Category: HOUSE ___
  Clues: ['lines', 'outs', 'line', 'table']  |  Category: TIME ___
  Clues: ['marks', 'shop', 'store', 'keeping']  |  Category: BOOK ___

Example for 'MAN':
  Clues: ['kirk', 'port', 'hunts', 'wing']  |  Category: ___ MAN


In [25]:
import itertools

def generate_random_puzzles(vocab: set[str],
                            n: int = 1000,
                            max_attempts: int = 50_000,
                            rng: Optional[random.Random] = None) -> list[str]:
    rng = rng or random.Random()
    vocab_list = list(vocab)
    mechanisms = ["prefix", "suffix"]
    seen_keys: set[str] = set()
    puzzles: list[str] = []

    attempts = 0

    while len(puzzles) < n and attempts < max_attempts:
        attempts += 1
        base = rng.choice(vocab_list)
        mech = rng.choice(mechanisms)
        try:
            if mech == "prefix":
                clues, category = generate_prefix_puzzle(base, vocab, n_clues=4, rng=rng)
                key = f"PREFIX|{category}|" + "|".join(sorted(clues))
                desc = f"PREFIX  ___ {category.upper()} : " + ", ".join(clues)
            else:
                clues, category = generate_suffix_puzzle(base, vocab, n_clues=4, rng=rng)
                key = f"SUFFIX|{category}|" + "|".join(sorted(clues))
                desc = f"SUFFIX  {category.upper()} ___ : " + ", ".join(clues)
        except ValueError:
            continue

        if key in seen_keys:
            continue

        seen_keys.add(key)
        puzzles.append(desc)

    return puzzles

rng = random.Random(42)
puzzles = generate_random_puzzles(vocab, n=1000, rng=rng)

print(f"Generated {len(puzzles)} puzzles:\n")

for i, p in enumerate(puzzles, start=1):
    print(f"{i:4d}. {p}")

Generated 576 puzzles:

   1. SUFFIX  RAIN ___ : bow, coat, forests, storm
   2. SUFFIX  COB ___ : ham, webs, alt, urn
   3. PREFIX  ___ RTS : dive, shi, ale, cha
   4. SUFFIX  AIR ___ : mail, men, show, strike
   5. PREFIX  ___ BOROUGH : hills, peter, gains, scar
   6. PREFIX  ___ ICK : homes, mag, mal, carr
   7. SUFFIX  RET ___ : rain, old, ina, raining
   8. SUFFIX  INFOR ___ : mal, ming, mer, med
   9. PREFIX  ___ RIES : fer, car, dai, surge
  10. PREFIX  ___ RALLY : cent, gene, late, lite
  11. PREFIX  ___ ONES : ram, every, some, overt
  12. SUFFIX  SPECI ___ : ally, als, mens, men
  13. SUFFIX  BUTT ___ : ers, hurt, ons, ing
  14. SUFFIX  TURN ___ : tables, table, pike, around
  15. SUFFIX  SCA ___ : red, thing, res, les
  16. SUFFIX  HARD ___ : ball, wick, working, back
  17. SUFFIX  DRAW ___ : bridge, ing, back, ers
  18. PREFIX  ___ NDS : bra, conte, liga, hou
  19. SUFFIX  INFO ___ : graphics, wars, graphic, rms
  20. SUFFIX  FIT ___ : ting, ted, ter, bit
  21. PREFIX  ___ 

In [26]:
%pip install pronouncing

Note: you may need to restart the kernel to use updated packages.


In [27]:
import pronouncing

def find_rhyme_group(base_word: str,
                     vocab: set[str],
                     min_group_size: int = 4,
                     min_zipf: float = MIN_ZIPF_CLUE) -> list[str]:
    
    base = clean_token(base_word)

    if zipf_frequency(base, "en") < min_zipf:
        return []

    rhyme_candidates = set(pronouncing.rhymes(base))
    candidates = rhyme_candidates & vocab
    candidates.discard(base)

    filtered = [w for w in candidates if zipf_frequency(w, "en") >= min_zipf]
    filtered = sorted(set(filtered))

    if len(filtered) < min_group_size:
        return []

    return filtered

def generate_rhyme_puzzle(base_word: str,
                          vocab: set[str],
                          n_clues: int = 4,
                          rng: Optional[random.Random] = None) -> tuple[list[str], str]:
    rng = rng or random
    group = find_rhyme_group(base_word, vocab, min_group_size=n_clues)

    if not group:
        raise ValueError(f"No sufficient rhyme group found for base word '{base_word}'")

    clues = rng.sample(group, n_clues)
    return clues, base_word



def generate_random_rhyme_puzzles(vocab: set[str],
                                  n: int = 100,
                                  max_attempts: int = 20_000,
                                  rng: Optional[random.Random] = None) -> list[str]:

    rng = rng or random.Random()
    vocab_list = list(vocab)
    seen_keys: set[str] = set()
    puzzles: list[str] = []

    attempts = 0

    while len(puzzles) < n and attempts < max_attempts:
        attempts += 1
        base = rng.choice(vocab_list)

        try:
            clues, category = generate_rhyme_puzzle(base, vocab, n_clues=4, rng=rng)

        except ValueError:
            continue

        key = f"RHYME|{category}|" + "|".join(sorted(clues))

        if key in seen_keys:
            continue

        seen_keys.add(key)
        desc = f"RHYME  (clues rhyme with {category.upper()}): " + ", ".join(clues)
        puzzles.append(desc)

    return puzzles


In [28]:
rng = random.Random(123)
rhyme_puzzles = generate_random_rhyme_puzzles(vocab, n=100, rng=rng)

print(f"Generated {len(rhyme_puzzles)} rhyme-based puzzles:\n")

for i, p in enumerate(rhyme_puzzles, start=1):
    print(f"{i:4d}. {p}")

Generated 100 rhyme-based puzzles:

   1. RHYME  (clues rhyme with WASTELAND): inland, parkland, heartland, broadband
   2. RHYME  (clues rhyme with NARCISSISTIC): mystic, autistic, antagonistic, linguistic
   3. RHYME  (clues rhyme with LANCE): romance, france, vance, glance
   4. RHYME  (clues rhyme with MARIA): tia, eritrea, dia, ria
   5. RHYME  (clues rhyme with PLASTIC): elastic, sarcastic, fantastic, scholastic
   6. RHYME  (clues rhyme with THEO): neo, leo, geo, yeo
   7. RHYME  (clues rhyme with SIMPLIFY): butterfly, samurai, nikolai, certify
   8. RHYME  (clues rhyme with OSAMA): obama, comma, yokohama, sama
   9. RHYME  (clues rhyme with FANTASTIC): stochastic, drastic, plastic, enthusiastic
  10. RHYME  (clues rhyme with OUR): sour, devour, hour, bower
  11. RHYME  (clues rhyme with ADVERSITY): cory, maggie, identity, catchy
  12. RHYME  (clues rhyme with ORTON): shorten, norton, horton, morton
  13. RHYME  (clues rhyme with INDONESIAN): lesion, cohesion, tunisian, adhesion

In [ ]:
import json
from typing import Optional

def build_before_after_categories(vocab: set[str],
                                  max_categories: int = 800,
                                  rng: Optional[random.Random] = None) -> list[dict]:
    rng = rng or random.Random(100)
    vocab_list = list(vocab)
    groups: list[dict] = []
    seen_keys: set[str] = set()
    attempts = 0
    max_attempts = 200_000

    while len(groups) < max_categories and attempts < max_attempts:
        attempts += 1
        base = rng.choice(vocab_list)
        mechanism = rng.choice(["prefix", "suffix"])

        try:
            if mechanism == "prefix":
                clues, category = generate_prefix_puzzle(base, vocab, n_clues=4, rng=rng)
                kind = "before_prefix"
                group_name = f"WORDS THAT CAN PRECEDE {category.upper()}"
                combined = [f"{c.upper()}{category.upper()}" for c in clues]
                explanation = ", ".join(
                    f"{c.upper()}+{category.upper()} = {combo}"
                    for c, combo in zip(clues, combined)
                )
            else:
                clues, category = generate_suffix_puzzle(base, vocab, n_clues=4, rng=rng)
                kind = "after_suffix"
                group_name = f"WORDS THAT CAN FOLLOW {category.upper()}"
                combined = [f"{category.upper()}{c.upper()}" for c in clues]
                explanation = ", ".join(
                    f"{category.upper()}+{c.upper()} = {combo}"
                    for c, combo in zip(clues, combined)
                )

        except ValueError:
            continue

        key = f"{kind}|{category.lower()}|" + "|".join(sorted(w.lower() for w in clues))
        if key in seen_keys:
            continue
        seen_keys.add(key)

        entry = {
            "group": group_name,
            "members": [w.upper() for w in clues],
            "explanation": explanation,
            "type": kind,
        }
        groups.append(entry)

    return groups

def build_rhyme_categories(vocab: set[str],
                           max_categories: int = 800,
                           rng: Optional[random.Random] = None) -> list[dict]:
    rng = rng or random.Random(200)
    vocab_list = list(vocab)
    groups: list[dict] = []
    seen_keys: set[str] = set()
    attempts = 0
    max_attempts = 200_000

    while len(groups) < max_categories and attempts < max_attempts:
        attempts += 1
        base = rng.choice(vocab_list)
        try:
            clues, category = generate_rhyme_puzzle(base, vocab, n_clues=4, rng=rng)
        except ValueError:
            continue

        key = f"rhyme|{category.lower()}|" + "|".join(sorted(w.lower() for w in clues))

        if key in seen_keys:
            continue
        seen_keys.add(key)

        group_name = f"WORDS THAT RHYME WITH {category.upper()}"
        explanation = f"{', '.join(w.upper() for w in clues)} all rhyme with {category.upper()}"
        
        entry = {
            "group": group_name,
            "members": [w.upper() for w in clues],
            "explanation": explanation,
            "type": "rhyme",
        }

        groups.append(entry)

    return groups

before_after_categories = build_before_after_categories(vocab, max_categories=800)
rhyme_categories = build_rhyme_categories(vocab, max_categories=800)

all_categories = before_after_categories + rhyme_categories
output_path = "/Users/andrewwang/Desktop/generated_before_after_and_rhyme_categories.json"

with open(output_path, "w") as f:
    json.dump(all_categories, f, indent=2)


print(f"Wrote {len(all_categories)} categories to {output_path}")

Wrote 1600 categories to /Users/andrewwang/Desktop/generated_before_after_and_rhyme_categories.json
